# Investigación de datos de "base.xlsx"

Columnas Criticas
**`Periodo`**: idetnificar y normalizar las fechas.                  
**`Neto`**: (negativos, ceros y atípicos) y la "integridad referencial".

In [10]:
import re
import sys
from collections import Counter
from datetime import date, datetime
from pathlib import Path

import openpyxl
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "Base.xlsx").exists():
    RAIZ = RAIZ.parent

pd.set_option("display.width", 120)
ARCHIVO = RAIZ / "Base.xlsx"
ventas = pd.read_excel(ARCHIVO, sheet_name="Ventas", dtype=str)   
print(f"{len(ventas):,} filas de ventas cargadas")

ModuleNotFoundError: No module named 'clean'

### validación de diferentes tipos de datos en la columna Periodo
bloque para conocer las diferencias y cantidades

In [7]:
for x, cantidad in ventas["Periodo"].dropna().astype(str).value_counts().sort_index().items():
    print(f"{repr(x)} -> {cantidad}")

'2026 06' -> 70305
'2026-04' -> 70867
'2026.01' -> 82387
'2026.03' -> 79892
'2026/05' -> 72509
'2026_02' -> 70782


### Identificando las diferencias normalicemos todas las fechas:

In [ ]:
_PERIODO = re.compile(r"(\d{4})\D+?(\d{1,2})")


def norm_periodo_local(valor):
    if valor is None:
        return None
    if isinstance(valor, (datetime, date)):
        return f"{valor.year:04d}-{valor.month:02d}"
    m = _PERIODO.search(str(valor))
    if not m:
        return None
    year, mes = int(m.group(1)), int(m.group(2))
    return f"{year:04d}-{mes:02d}" if 1 <= mes <= 12 else None


ventas["periodo"] = ventas["Periodo"].map(norm_periodo_local)
print(ventas["periodo"].value_counts().sort_index().to_string())
print("periodos:", ventas['periodo'].nunique(), "| nulos:", int(ventas['periodo'].isna().sum()))

periodo
2026-01    82387
2026-02    70782
2026-03    79892
2026-04    70867
2026-05    72509
2026-06    70305
periodos: 6 | nulos: 0
